# 01 — Frame inventory

Header-only census of `Calibrated_Lights/` (APP‑calibrated, debayered RGB
FITS).  Produces `products/frame_inventory.csv` with, per frame: filter‑wheel
position, exposure, camera timestamp and its UTC conversion.

Legacy source: `categorize_data.ipynb`, `find_sun_center.ipynb` cell 6.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
rows = []
for f in utils.list_calibrated_frames():
    h = utils.read_header(f)
    rows.append(dict(filename=f,
                     position=utils.position_from_filename(f),
                     exposure_time=h["EXPTIME"],
                     inverse_exposure_time=utils.inverse_exposure(h),
                     date_obs_camera=h["DATE-OBS"],
                     date_obs_utc=utils.fits_timestamp_utc(h)))
inv = pd.DataFrame(rows)
inv.to_csv(config.PRODUCTS_DIR / "frame_inventory.csv", index=False)
print(len(inv), "frames")
inv.head()

In [ ]:
# Frames per (position, exposure).  Position 4 = unpolarised slot, not used further.
pd.crosstab(inv.inverse_exposure_time, inv.position, margins=True)

In [ ]:
# Time span of the polarimetric sequence (UTC)
pol = inv[inv.position.isin(config.POLARIZER_POSITIONS)]
print("first:", pol.date_obs_utc.min(), " last:", pol.date_obs_utc.max())

In [ ]:
# Quick look at one frame per exposure (Position 1), luminance, asinh stretch
from matplotlib.colors import Normalize
sub = pol[pol.position == "1"].drop_duplicates("inverse_exposure_time").sort_values("inverse_exposure_time", ascending=False)
fig, axes = plt.subplots(1, len(sub), figsize=(3*len(sub), 2.5))
for ax, (_, r) in zip(axes, sub.iterrows()):
    img = fits.getdata(config.CALIBRATED_LIGHTS_DIR / r.filename, memmap=True)
    L = utils.luminance(np.asarray(img[:, ::8, ::8], dtype=np.float32))
    ax.imshow(utils.asinh_stretch(L), cmap="gray"); ax.set_title(f"1/{r.inverse_exposure_time} s", fontsize=9); ax.axis("off")
plt.tight_layout()